In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [2]:
df = pd.read_csv("../data/raw/loan_data.csv")
X = df.drop(columns=["loan_status"])
y = df["loan_status"]

In [3]:
X["loan_to_income"] = (
    X["loan_amnt"] / X["person_income"]
)

In [7]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

In [8]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [9]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [11]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(max_iter=1000)
        )
    ]
)

In [12]:
knn_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            KNeighborsClassifier(n_neighbors=5)
        )
    ]
)

In [13]:
nb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GaussianNB()
        )
    ]
)

In [14]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [15]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

In [16]:
logistic_cv = cross_validate(
    logistic_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

In [17]:
print(
    "Logistic Regression"
)

for metric in scoring:
    scores = logistic_cv[
        f"test_{metric}"
    ]

    print(
        f"{metric}: "
        f"{scores.mean():.4f} "
        f"+/- {scores.std():.4f}"
    )

Logistic Regression
accuracy: 0.8963 +/- 0.0018
precision: 0.7762 +/- 0.0063
recall: 0.7495 +/- 0.0045
f1: 0.7626 +/- 0.0035
roc_auc: 0.9541 +/- 0.0016


In [18]:
knn_cv = cross_validate(
    knn_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

In [19]:
print("KNN")

for metric in scoring:
    scores = knn_cv[
        f"test_{metric}"
    ]

    print(
        f"{metric}: "
        f"{scores.mean():.4f} "
        f"+/- {scores.std():.4f}"
    )

KNN
accuracy: 0.8958 +/- 0.0011
precision: 0.7953 +/- 0.0035
recall: 0.7152 +/- 0.0070
f1: 0.7531 +/- 0.0035
roc_auc: 0.9289 +/- 0.0022


In [20]:
nb_cv = cross_validate(
    nb_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

In [21]:
print("Gaussian Naive Bayes")

for metric in scoring:
    scores = nb_cv[
        f"test_{metric}"
    ]

    print(
        f"{metric}: "
        f"{scores.mean():.4f} "
        f"+/- {scores.std():.4f}"
    )

Gaussian Naive Bayes
accuracy: 0.7348 +/- 0.0021
precision: 0.4559 +/- 0.0020
recall: 0.9978 +/- 0.0004
f1: 0.6258 +/- 0.0018
roc_auc: 0.9371 +/- 0.0025


In [22]:
cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "KNN",
        "Gaussian Naive Bayes"
    ],

    "Accuracy": [
        logistic_cv["test_accuracy"].mean(),
        knn_cv["test_accuracy"].mean(),
        nb_cv["test_accuracy"].mean()
    ],

    "Precision": [
        logistic_cv["test_precision"].mean(),
        knn_cv["test_precision"].mean(),
        nb_cv["test_precision"].mean()
    ],

    "Recall": [
        logistic_cv["test_recall"].mean(),
        knn_cv["test_recall"].mean(),
        nb_cv["test_recall"].mean()
    ],

    "F1": [
        logistic_cv["test_f1"].mean(),
        knn_cv["test_f1"].mean(),
        nb_cv["test_f1"].mean()
    ],

    "ROC-AUC": [
        logistic_cv["test_roc_auc"].mean(),
        knn_cv["test_roc_auc"].mean(),
        nb_cv["test_roc_auc"].mean()
    ]
})

In [23]:
cv_results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.896289,0.776179,0.7495,0.762583,0.954076
1,KNN,0.895800,0.795314,0.7152,0.753105,0.928899
2,Gaussian Naive Bayes,0.734844,0.455875,0.9978,0.625821,0.937083


In [24]:
cv_results.set_index("Model").idxmax()

Accuracy      Logistic Regression
Precision                     KNN
Recall       Gaussian Naive Bayes
F1            Logistic Regression
ROC-AUC       Logistic Regression
dtype: object

In [25]:
cv_stability = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "KNN",
        "Gaussian Naive Bayes"
    ],

    "F1 Mean": [
        logistic_cv["test_f1"].mean(),
        knn_cv["test_f1"].mean(),
        nb_cv["test_f1"].mean()
    ],

    "F1 Std": [
        logistic_cv["test_f1"].std(),
        knn_cv["test_f1"].std(),
        nb_cv["test_f1"].std()
    ],

    "ROC-AUC Mean": [
        logistic_cv["test_roc_auc"].mean(),
        knn_cv["test_roc_auc"].mean(),
        nb_cv["test_roc_auc"].mean()
    ],

    "ROC-AUC Std": [
        logistic_cv["test_roc_auc"].std(),
        knn_cv["test_roc_auc"].std(),
        nb_cv["test_roc_auc"].std()
    ]
})

cv_stability

,Model,F1 Mean,F1 Std,ROC-AUC Mean,ROC-AUC Std
0,Logistic Regression,0.762583,0.003481,0.954076,0.001602
1,KNN,0.753105,0.003515,0.928899,0.002184
2,Gaussian Naive Bayes,0.625821,0.001840,0.937083,0.002517
